In [48]:
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support

%pip install torch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

Note: you may need to restart the kernel to use updated packages.


In [88]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim=8, hidden_layers=1, dropout=0.0):
        super().__init__()

        if hidden_layers == 1:
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, bottleneck_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
            self.decoder = nn.Sequential(
                nn.Linear(bottleneck_dim, input_dim)
            )

        elif hidden_layers == 2:
            hidden_dim = max(input_dim // 2, bottleneck_dim + 2)
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, bottleneck_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            )
            self.decoder = nn.Sequential(
                nn.Linear(bottleneck_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, input_dim)
            )
        else:
            raise ValueError("hidden_layers must be 1 or 2")

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

In [89]:
# Add Gaussian noise during training to make the autoencoder more robust
def add_noise(x: torch.Tensor, noise_std: float = 0.05) -> torch.Tensor:
    noise = torch.randn_like(x) * noise_std
    return x + noise

# Create PyTorch DataLoader for autoencoder, where input = target
def make_loader(X_array, batch_size: int = 64, shuffle: bool = False) -> DataLoader:
    if hasattr(X_array, "to_numpy"):
        X_array = X_array.to_numpy()

    X_array = X_array.astype("float32")
    tensor_x = torch.from_numpy(X_array)
    dataset = TensorDataset(tensor_x, tensor_x)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

- Reconstruction loss: MSE
- Early stopping

In [90]:
import copy
import numpy as np
import torch
import torch.nn as nn

def train_autoencoder(
    model,
    train_loader,
    val_loader,
    learning_rate=1e-3,
    max_epochs=100,
    noise_std=0.05,
    patience=10,
    device="cpu"
):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())   # initialize safely
    wait = 0

    for epoch in range(max_epochs):
        # ----- train -----
        model.train()
        for xb, _ in train_loader:
            xb = xb.to(device)

            noisy_xb = xb + noise_std * torch.randn_like(xb)
            recon = model(noisy_xb)
            loss = criterion(recon, xb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # ----- validate -----
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, _ in val_loader:
                xb = xb.to(device)
                recon = model(xb)
                val_loss = criterion(recon, xb)
                val_losses.append(val_loss.item())

        if len(val_losses) == 0:
            raise ValueError("val_loader is empty, so validation loss cannot be computed.")

        mean_val_loss = np.mean(val_losses)

        if np.isnan(mean_val_loss):
            raise ValueError("Validation loss is NaN. Check your input data for NaN/inf values.")

        if mean_val_loss < best_val_loss:
            best_val_loss = mean_val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    model.load_state_dict(best_state)
    return model, best_val_loss

In [91]:
# return sample-level anomaly scores and feature-level squared errors
def get_reconstruction_errors(model, X_array, device: str = "cpu"):
    model.eval()

    # convert DataFrame -> NumPy
    if hasattr(X_array, "to_numpy"):
        X_array = X_array.to_numpy()

    X_array = X_array.astype("float32")
    X_tensor = torch.from_numpy(X_array).to(device)

    with torch.no_grad():
        X_hat = model(X_tensor)

    sq_error = (X_tensor - X_hat) ** 2
    row_errors = sq_error.mean(dim=1).cpu().numpy()      # anomaly score per row
    feature_errors = sq_error.cpu().numpy()              # feature-level errors
    recon = X_hat.cpu().numpy()

    return row_errors, feature_errors, recon

**Full training pipeline:**
1. split data
2. scale on normal training data only
3. train AE on normal training data
4. calibrate threshold on normal validation data
5. evaluate on full test set

In [92]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import pandas as pd

In [93]:
def main_autoencoder(
    df: pd.DataFrame,
    normal_label: int = 0,
    test_size: float = 0.2,
    val_size_on_normal: float = 0.2,
    batch_size: int = 64,
    bottleneck_dim: int = 8,
    hidden_layers: int = 2,
    dropout: float = 0.1,
    learning_rate: float = 1e-3,
    max_epochs: int = 100,
    noise_std: float = 0.05,
    patience: int = 10,
    threshold_method: str = "percentile",
    threshold_quantile: float = 85,
    # threshold_k: float = 3.0,
    device: str = "cpu"
):
    X = df.drop(columns=["loan_status"])
    y = df["loan_status"]

    # categorical + numeric columns
    cat_cols = ["person_home_ownership", "loan_intent", "loan_grade"]
    num_cols = [col for col in X.columns if col not in cat_cols]

    # np.log1p is better than np.log because it handles zero values safely
    log_transformer = FunctionTransformer(
        np.log1p, 
        validate=True, 
        feature_names_out="one-to-one"  
    )

    # Pipeline for numeric columns: Log -> Scale
    num_pipeline = Pipeline([
        ("log", log_transformer),
        ("scaler", StandardScaler())
    ])

    # Integrate into ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipeline, num_cols), # Applies Log then Scale
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
        ]
    )

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=42,
        stratify=y
    )
    
    # Fit Preprocessor on Training Data and Transform all sets
    X_trainval_proc = preprocessor.fit_transform(X_trainval)
    X_test_proc = preprocessor.transform(X_test)
    
    # Get feature names after OHE expansion
    feature_names = preprocessor.get_feature_names_out().tolist()

    # Normal-only subset for AE training
    X_trainval_normal = X_trainval_proc[y_trainval == normal_label].copy()

    # Normal train-validation split
    X_train_normal, X_val_normal = train_test_split(
        X_trainval_normal,
        test_size=val_size_on_normal,
        random_state=42
    )

    # print(X_train_normal.shape)
    # print(X_val_normal.shape)

    # DataLoaders
    train_loader = make_loader(X_train_normal, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(X_val_normal, batch_size=batch_size, shuffle=False)

    # Build model
    input_dim = X_train_normal.shape[1]
    model = DenoisingAutoencoder(
        input_dim=input_dim,
        bottleneck_dim=bottleneck_dim,
        hidden_layers=hidden_layers,
        dropout=dropout
    )


    model, best_val_loss = train_autoencoder(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        learning_rate=learning_rate,
        max_epochs=max_epochs,
        noise_std=noise_std,
        patience=patience,
        device=device
    )

    # Validation-normal errors for threshold calibration
    val_normal_errors, _, _ = get_reconstruction_errors(model, X_val_normal, device=device)

    threshold = np.percentile(val_normal_errors, threshold_quantile)
    
    # if threshold_method == "percentile":
    #     threshold = np.percentile(val_normal_errors, threshold_quantile)
    # elif threshold_method == "mu_sigma":
    #     threshold = val_normal_errors.mean() + threshold_k * val_normal_errors.std()
    # else:
    #     raise ValueError("threshold_method must be 'percentile' or 'mu_sigma'")

    # Final evaluation on test set
    test_errors, test_feature_errors, _ = get_reconstruction_errors(model, X_test_proc, device=device)
    y_pred_anomaly = (test_errors > threshold).astype(int)

    roc_auc = roc_auc_score(y_test, test_errors)
    pr_auc = average_precision_score(y_test, test_errors)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred_anomaly, average="binary", zero_division=0
    )

    metrics = {
        "best_val_loss": float(best_val_loss),
        "threshold": float(threshold),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "flag_rate": float(y_pred_anomaly.mean())
    }
    
    scaler = StandardScaler()

    artifacts = {
        "model": model,
        "preprocessor": preprocessor,
        "threshold": threshold,
        "feature_names": feature_names,
        "model_config": {
            "input_dim": input_dim,
            "bottleneck_dim": bottleneck_dim,
            "hidden_layers": hidden_layers,
            "dropout": dropout
        },
        "metrics": metrics
    }

    return artifacts

In [94]:
# save trained model, scaler, threshold, feature names, and config
def save_anomaly_artifacts(artifacts: dict, artifact_dir: str) -> None:
    os.makedirs(artifact_dir, exist_ok=True)

    # Save PyTorch weights only
    torch.save(
        artifacts["model"].state_dict(),
        os.path.join(artifact_dir, "autoencoder_model.pt")
    )

    # Save ColumnTransformer (which includes the scaler and OHE)
    with open(os.path.join(artifact_dir, "preprocessor.pkl"), "wb") as f:
        pickle.dump(artifacts["preprocessor"], f)

    # Save metadata needed for inference
    metadata = {
        "threshold": artifacts["threshold"],
        "feature_names": artifacts["feature_names"],
        "model_config": artifacts["model_config"],
        "metrics": artifacts["metrics"]
    }
    with open(os.path.join(artifact_dir, "metadata.pkl"), "wb") as f:
        pickle.dump(metadata, f)
        
    print(f"Artifacts successfully saved to: {artifact_dir}")

# example call

In [95]:
device = "cuda" if torch.cuda.is_available() else "cpu"


In [96]:
df = pd.read_csv("credit_risk_dataset.csv") 
# drop last tow columns which are not features
df = df.iloc[:, :-2]
df = df.dropna()
df.columns.tolist()

['person_age',
 'person_income',
 'person_home_ownership',
 'person_emp_length',
 'loan_intent',
 'loan_grade',
 'loan_amnt',
 'loan_int_rate',
 'loan_status',
 'loan_percent_income']

In [97]:
# # fit and transform
# X_processed = preprocessor.fit_transform(X)

# feature_names = preprocessor.get_feature_names_out()

# X_processed_df = pd.DataFrame(
#     X_processed,
#     columns=feature_names,
#     index=X.index
# )

# X_processed_df.columns.tolist()

In [98]:
artifacts = main_autoencoder(
    df,    
    normal_label=0,             # 0 means normal applicant
    bottleneck_dim=8,
    hidden_layers=2,
    dropout=0.1,
    learning_rate=1e-3,
    max_epochs=100,
    noise_std=0.05,
    patience=10,
    threshold_method="percentile",
    threshold_quantile=85,
    device=device
)

print(artifacts["metrics"])

{'best_val_loss': 0.08999720568719663, 'threshold': 0.11904595792293549, 'roc_auc': 0.7876320831583121, 'pr_auc': 0.5281605179972801, 'precision': 0.5070527097253155, 'recall': 0.5503626107977437, 'f1': 0.5278207109737248, 'flag_rate': 0.23516061452513967}


In [99]:
save_anomaly_artifacts(artifacts, artifact_dir="artifacts/ae_agent")

Artifacts successfully saved to: artifacts/ae_agent
